# Which level is this text? Una regressione sulla leggibilità
**Script intelligenti: sviluppare modelli di Machine Learning con il codice**

I dati: **OneStopEnglish** (Vajjala e Lučić 2018, licenza CC BY-SA 4.0): 189 articoli del Guardian riscritti da
insegnanti in tre livelli, elementary, intermediate e advanced, per studenti adulti di inglese. Per ogni testo il
file ha già contato parole, frasi e sillabe.

La domanda: **dalla lunghezza di frasi e parole si può predire il livello di un testo?** È una regressione: la label
è un numero (1, 2, 3).

Oggi non scriviamo codice a mano: lo scrive **Gemini**, l'assistente di Colab. Il nostro lavoro è chiedere bene e
controllare il risultato.

| | |
|---|---|
| **1 · Chiedi chiaro** | che cosa vuoi, con quali dati, in che forma vuoi il risultato |
| **2 · Leggi** | il codice proposto, anche senza capirlo tutto: quali dati usa? che cosa calcola? |
| **3 · Esegui** | e guarda che cosa succede, errori compresi |
| **4 · Controlla** | il risultato torna con quello che sai? I numeri sono plausibili? |

## Come chiedere a Gemini
1. In una cella di codice vuota, cliccate su **genera** (oppure aprite il pannello di Gemini, in alto a destra).
2. Scrivete la richiesta in italiano: potete copiare il prompt di esempio.
3. Leggete il codice proposto prima di accettarlo.
4. Eseguitelo con ▶ e guardate il risultato.

*Se Gemini non è disponibile:* usate ChatGPT o Claude in un'altra scheda e incollate qui il codice.

## Preparazione
Eseguite questa cella con ▶: scarica i dati. Se il download non funziona, caricate i file a mano dal pannello **File** (l'icona a forma di cartella a sinistra).

In [ ]:
# Scarica i file dei dati nella cartella di lavoro di Colab.
import os
import urllib.request

if not os.path.exists("livelli_inglese.csv"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/matteolimoncini/script-intelligenti/main/incontro-3/dati/livelli_inglese.csv", "livelli_inglese.csv")
print("File pronti:", "livelli_inglese.csv")

## Tappa 1 · Guardare i testi
Ogni riga del file è un testo: lo stesso articolo compare tre volte, una per livello. Le colonne
**parole_per_frase** e **sillabe_per_parola** sono le feature; **livello_num** è la label (1 elementary,
2 intermediate, 3 advanced); **flesch** è il Flesch Reading Ease, la formula che Word usa per la leggibilità
(più è alto, più il testo è facile).

**Prompt di esempio:**

> Carica livelli_inglese.csv con pandas. Quanti testi ci sono per livello? Mostrami la media di parole per frase,
> sillabe per parola e flesch per ogni livello, nell'ordine elementary, intermediate, advanced.

In [ ]:
# Chiedete a Gemini qui sotto (clic su «genera»)

**Domande**
- Le frasi si allungano salendo di livello? E le parole?
- Il Flesch sale o scende con il livello? Perché?

## Tappa 2 · Il Flesch mette i testi in ordine?
**Prompt di esempio:**

> Fai un boxplot del flesch per livello, nell'ordine elementary, intermediate, advanced. Poi, per ogni articolo,
> controlla se il flesch scende da elementary a intermediate ad advanced e dimmi in quanti articoli succede.

In [ ]:
# Chiedete a Gemini qui sotto (clic su «genera»)

**Domande**
- Con lo stesso articolo, il Flesch quasi sempre mette i tre livelli in ordine. E tra articoli diversi?
- Un testo elementary può avere un Flesch più basso di un testo advanced? Che cosa vuol dire per chi sceglie un testo
  per la classe?

## Tappa 3 · Una regressione sul livello
**Prompt di esempio:**

> Addestra una regressione lineare che predice livello_num da parole_per_frase e sillabe_per_parola. Dividi in
> addestramento e test per articolo, con GroupShuffleSplit (test_size=0.2, random_state=42), così le tre versioni
> dello stesso articolo stanno dalla stessa parte. Dimmi MAE e R² sul test e il MAE di chi prevede sempre la media.

In [ ]:
# Chiedete a Gemini qui sotto (clic su «genera»)

**Poi:**

> Aggiungi la colonna parole (la lunghezza del testo) alle feature e rifai gli stessi conti.

**Domande**
- Perché dividere per articolo? Che cosa succederebbe se la versione elementary di un articolo fosse
  nell'addestramento e quella advanced nel test?
- Il modello batte la media? Di quanto?
- Con la lunghezza del testo migliora molto. È un segnale vero di difficoltà, o una caratteristica di questo corpus
  (le versioni advanced sono più lunghe)? Funzionerebbe sui testi del vostro libro?

## Tappa 4 · I vostri testi
La cella qui sotto è già pronta: conta le feature di un testo che incollate voi e chiede il livello a un
modello addestrato su tutto il file. Usate testi del libro con il livello già indicato, mai testi scritti dagli
studenti.

In [ ]:
# Chiedete a Gemini qui sotto (clic su «genera»)

Eseguite la cella con ▶ dopo aver incollato il vostro testo.

In [ ]:
import re

def sillabe(parola):
    """Conta le sillabe di una parola inglese con una regola semplice: i gruppi di vocali, meno la «e» muta finale."""
    p = re.sub(r"[^a-z]", "", parola.lower())
    if not p:
        return 0
    gruppi = len(re.findall(r"[aeiouy]+", p))
    if p.endswith("e") and not p.endswith(("le", "ee", "ye")) and gruppi > 1:
        gruppi -= 1
    return max(gruppi, 1)

def feature(testo):
    """Le stesse feature del file: parole per frase, sillabe per parola e il Flesch Reading Ease."""
    frasi = [f for f in re.split(r"(?<=[.!?])\s+|\n+", testo.strip()) if re.search(r"[A-Za-z]", f)]
    parole = re.findall(r"[A-Za-z]+(?:['’][A-Za-z]+)?", testo)
    pf = len(parole) / len(frasi)
    sp = sum(sillabe(p) for p in parole) / len(parole)
    return {"parole_per_frase": round(pf, 2), "sillabe_per_parola": round(sp, 3),
            "flesch": round(206.835 - 1.015 * pf - 84.6 * sp, 1)}

import pandas as pd
from sklearn.linear_model import LinearRegression

# un modello addestrato su tutti i testi del file, con le due feature della tappa 3
dati = pd.read_csv("livelli_inglese.csv")
modello = LinearRegression().fit(dati[["parole_per_frase", "sillabe_per_parola"]], dati["livello_num"])

# incollate qui un testo del vostro libro (tra le tre virgolette), con il livello che il libro indica
mio_testo = """
Paste your text here. It should be at least a few sentences long, so that the counts make sense.
"""

f = feature(mio_testo)
livello = modello.predict(pd.DataFrame([f])[["parole_per_frase", "sillabe_per_parola"]])[0]
print(f)
print("Livello previsto:", round(livello, 2), " (1 = elementary, 2 = intermediate, 3 = advanced)")
if not 0.5 <= livello <= 3.5:
    print("Fuori dai livelli del corpus: il testo è molto diverso da quelli di addestramento (estrapolazione).")

**Domande**
- Il livello previsto è vicino a quello del libro?
- Che cosa non vede il modello? (il vocabolario, la grammatica, l'argomento, quanto il testo è familiare a chi
  legge)

## Per finire
Il Flesch e la nostra regressione vedono solo quanto sono lunghe le frasi e le parole. Mettono quasi sempre in ordine
le tre versioni dello stesso articolo, ma tra testi diversi sbagliano spesso: il livello dipende anche da vocabolario,
grammatica e argomento. **In classe:** gli studenti possono calcolare a mano le feature di due testi, confrontare il
Flesch con il livello del libro e discutere che cosa la formula non vede.